# Current MOT and pMOT Force / Force-Gradient Overlays

This notebook recomputes the current on-axis radiation-pressure force and same-axis spatial force gradient for four models:

1. the conventional 24-state multilevel MOT;
2. `mot_strange`, using the corrected intensity-derived surrogate magnetic field;
3. the effective full scalar/vector/tensor `pMOT_2Level`; and
4. the controlled vector-only `pmot_vec_2Level`.

Every calculation uses a stationary atom and excludes gravity so that the radiation-pressure force laws are compared directly. The force-gradient row is calculated numerically from the exact force samples plotted immediately above it. The pMOT branches remain provisional diagnostics, not validated physical pMOT predictions.

## 1. Locate the repository and import the plotting implementation

This section makes the notebook independent of its launch directory and imports the same live functions used by `scripts/plot_updated_force_gradient_overlays.py`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / 'pyproject.toml').is_file() and (PROJECT_ROOT / 'src' / 'pmot').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError('Could not locate the pMOT repository root.')

for path in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from scripts import plot_updated_force_gradient_overlays as overlay

display(Markdown(f'Using repository root: `{PROJECT_ROOT}`'))

## 2. User controls: physics, scan range, resolution, and output location

Edit this cell before running the notebook. Multilevel settings apply to both the conventional MOT and `mot_strange`. The pMOT cooling and trapping powers apply to both the full and vector-only two-level pMOT branches. Set `RECOMPUTE_PHYSICS=False` to redisplay saved results without repeating the force calculation.

In [ ]:
# Shared laser detuning
COOLING_DETUNING_MHZ = -15.0

# Conventional multilevel MOT and mot_strange light powers
MULTILEVEL_COOLING_POWER_MW = 27.0
MULTILEVEL_REPUMP_POWER_MW = 0.1

# Full and vector-only effective two-level pMOT powers
PMOT_COOLING_POWER_MW = 1.0571
PMOT_TRAPPING_PATH_POWERS_MW = (31.524310589374092,) * 3  # (Px, Py, Pz)

# Symmetric on-axis position range and derivative-grid resolution
SCAN_LIMIT_MM = 12.5
BROAD_SAMPLES = 301
CENTER_SAMPLES = 161
FOCUS_SAMPLES = 201
FIGURE_DPI = 220

# Reuse the CSV after the first complete run when only replotting is needed.
RECOMPUTE_PHYSICS = True
OUTPUT_DIRECTORY = (
    PROJECT_ROOT / 'outputs' / 'diagnostics' / 'updated_force_gradient_overlays_notebook'
)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
DATA_PATH = OUTPUT_DIRECTORY / 'current_model_on_axis_force_gradient_linecuts.csv'
METADATA_PATH = OUTPUT_DIRECTORY / 'notebook_settings.json'

controls = pd.Series({
    'cooling detuning [MHz]': COOLING_DETUNING_MHZ,
    'MOT and mot_strange cooling power [mW/component]': MULTILEVEL_COOLING_POWER_MW,
    'MOT and mot_strange repump power [mW/component]': MULTILEVEL_REPUMP_POWER_MW,
    'two-level pMOT cooling power [mW/component]': PMOT_COOLING_POWER_MW,
    'two-level pMOT trapping powers (x,y,z) [mW/path]': PMOT_TRAPPING_PATH_POWERS_MW,
    'scan limit [mm]': SCAN_LIMIT_MM,
    'recompute physics': RECOMPUTE_PHYSICS,
}, name='selected value')
display(controls.to_frame())

## 3. Apply the controls and construct the nonuniform position grid

The broad grid covers the full requested interval. Dense samples near the origin and the focused-beam regions improve the numerical derivative without forcing the entire scan to use the smallest spacing. The default $\pm12.5$ mm limit remains inside the current `mot_strange` hard cell-support boundary.

In [ ]:
overlay.SCAN_LIMIT_MM = float(SCAN_LIMIT_MM)
overlay.COOLING_DETUNING_MHZ = float(COOLING_DETUNING_MHZ)
overlay.MULTILEVEL_COOLING_POWER_MW = float(MULTILEVEL_COOLING_POWER_MW)
overlay.MULTILEVEL_REPUMP_POWER_MW = float(MULTILEVEL_REPUMP_POWER_MW)
overlay.PMOT_COOLING_POWER_MW = float(PMOT_COOLING_POWER_MW)
overlay.PMOT_TRAPPING_PATH_POWERS_MW = np.asarray(
    PMOT_TRAPPING_PATH_POWERS_MW, dtype=float
)
overlay.FIGURE_DPI = int(FIGURE_DPI)

scan_m = overlay.scan_coordinates_m(
    broad_samples=BROAD_SAMPLES,
    center_samples=CENTER_SAMPLES,
    focus_samples=FOCUS_SAMPLES,
)
print(f'{len(scan_m):,} unique positions per Cartesian axis')
print(f'Position range: {1e3 * scan_m[0]:g} to {1e3 * scan_m[-1]:g} mm')

## 4. Evaluate all four force laws and calculate their spatial gradients

This is the computationally expensive section. At every sampled point it evaluates the current model implementation, records the same-axis force, and then applies `numpy.gradient` to that same force curve. The resulting CSV is the common data source for every plot below.

In [ ]:
if RECOMPUTE_PHYSICS or not DATA_PATH.is_file():
    linecuts = overlay.evaluate_models(scan_m)
    linecuts.to_csv(DATA_PATH, index=False)
    print(f'Computed and saved {len(linecuts):,} rows to {DATA_PATH}')
else:
    linecuts = pd.read_csv(DATA_PATH)
    print(f'Reused {len(linecuts):,} saved rows from {DATA_PATH}')

display(linecuts.head())

## 5. Numerical check: force and force gradient at the origin

For symmetric light the radiation-pressure force should vanish at the origin. The force-gradient entries summarize the local restoring or anti-restoring slope: negative is restoring along the displayed coordinate.

In [ ]:
origin = linecuts[np.isclose(linecuts['coordinate_mm'], 0.0)].copy()
origin_summary = origin[[
    'model', 'axis', 'force_zN', 'force_gradient_1e19_n_per_m'
]].sort_values(['model', 'axis'])
display(origin_summary.style.format({
    'force_zN': '{:+.6g}',
    'force_gradient_1e19_n_per_m': '{:+.6g}',
}))

## 6. Plot A: conventional MOT, strange MOT, and full two-level pMOT

This 2-by-3 figure places $F_x,F_y,F_z$ in the top row and $dF_x/dx,dF_y/dy,dF_z/dz$ directly beneath them. It is the three-model comparison without the vector-only control.

In [ ]:
THREE_MODEL_FIGURE = OUTPUT_DIRECTORY / 'mot_strange_pmot2level_overlay.png'
overlay.plot_overlay(
    linecuts,
    ('mot_multilevel', 'mot_strange', 'pMOT_2Level'),
    title='Force and Force-Gradient Overlay: MOT, mot_strange, and 2-Level pMOT',
    output_path=THREE_MODEL_FIGURE,
)
display(Image(filename=str(THREE_MODEL_FIGURE)))

## 7. Plot B: all four current models

This is the complete overlay. The vector-only two-level pMOT is added as a separate color and line style so the effect of removing scalar and tensor shifts can be read directly.

In [ ]:
FOUR_MODEL_FIGURE = OUTPUT_DIRECTORY / 'mot_strange_pmot2level_vector_overlay.png'
overlay.plot_overlay(
    linecuts,
    ('mot_multilevel', 'mot_strange', 'pMOT_2Level', 'pmot_vec_2Level'),
    title='Force and Force-Gradient Overlay: All Four Current Models',
    output_path=FOUR_MODEL_FIGURE,
)
display(Image(filename=str(FOUR_MODEL_FIGURE)))

## 8. Plot C: full versus vector-only two-level pMOT

This isolates the role of the scalar-plus-tensor contribution while holding the two-level cooling and trapping powers fixed.

In [ ]:
PMOT_PAIR_FIGURE = OUTPUT_DIRECTORY / 'pmot2level_full_vs_vector_only_overlay.png'
overlay.plot_overlay(
    linecuts,
    ('pMOT_2Level', 'pmot_vec_2Level'),
    title='Force and Force-Gradient Overlay: Full vs Vector-Only 2-Level pMOT',
    output_path=PMOT_PAIR_FIGURE,
)
display(Image(filename=str(PMOT_PAIR_FIGURE)))

## 9. Save the selected settings and list every generated artifact

This final section records the notebook controls alongside the CSV and figures, making a later plot traceable to the physical and numerical settings that produced it.

In [ ]:
settings = {
    'stationary_atom': True,
    'gravity_excluded': True,
    'cooling_detuning_mhz': COOLING_DETUNING_MHZ,
    'multilevel_cooling_power_mw_per_component': MULTILEVEL_COOLING_POWER_MW,
    'multilevel_repump_power_mw_per_component': MULTILEVEL_REPUMP_POWER_MW,
    'pmot_cooling_power_mw_per_component': PMOT_COOLING_POWER_MW,
    'pmot_trapping_path_powers_mw_xyz': list(PMOT_TRAPPING_PATH_POWERS_MW),
    'scan_limit_mm': SCAN_LIMIT_MM,
    'sample_count_per_axis': int(len(scan_m)),
    'broad_samples': BROAD_SAMPLES,
    'center_samples': CENTER_SAMPLES,
    'focus_samples': FOCUS_SAMPLES,
    'figures': [str(THREE_MODEL_FIGURE), str(FOUR_MODEL_FIGURE), str(PMOT_PAIR_FIGURE)],
    'linecuts_csv': str(DATA_PATH),
}
METADATA_PATH.write_text(json.dumps(settings, indent=2) + '\n', encoding='utf-8')

artifacts = pd.DataFrame({
    'artifact': ['linecut data', 'three-model plot', 'four-model plot', 'pMOT pair plot', 'settings'],
    'path': [DATA_PATH, THREE_MODEL_FIGURE, FOUR_MODEL_FIGURE, PMOT_PAIR_FIGURE, METADATA_PATH],
})
display(artifacts)

## Interpretation boundary

The conventional and strange MOT curves use the replacement 24-state population-rate kernel. `mot_strange` contains no propagated 1529-nm trapping beams; it uses their remembered intensity geometry only to construct a surrogate magnetic field. The two effective two-level pMOT branches do propagate the trapping-light intensity geometry into differential transition shifts, but neither is an $m_F$-resolved Stark Hamiltonian. These plots diagnose local stationary force laws only and do not establish capture, loading, temperature, or dynamically stable trapping.